# E8c: grupos de features adicionales

Referencia: dos XGBoost con MPUD (uno para `y_f`, otro para `y_s`) con las 36 features podadas. Cada grupo se agrega solo y después todos juntos. Criterio fijado antes (`README.md`, E8c): AUC = mediana por persona dejando una persona fuera, contra la etiqueta del modelo, media de 5 semillas; un grupo se queda en un modelo si le suma 0.01 o más.

In [1]:
import os, json
from pathlib import Path
if Path.cwd().name == "notebooks":
    os.chdir("..")
import numpy as np
import pandas as pd
from experiments.modeling import load, feature_cols
from experiments import variants as V
from experiments import tabular as T
from smartuse.features_extra import GROUPS
SEEDS = [0, 1, 2, 3, 4]
TARGETS = ["y_f", "y_s"]

def with_extra(name):
    base = load(name)
    extra = pd.read_csv(f"data/processed/{name}_features_extra.csv")
    assert (base["t_utc"].values == extra["t_utc"].values).all()
    return pd.concat([base, extra.drop(columns=["user_id", "t_utc"])], axis=1)

real, mpud = with_extra("real"), with_extra("mpud")
base = V.pruned(feature_cols(load("real")))
configs = {"baseline": base, **{f"+ {g}": base + cols for g, cols in GROUPS.items()},
           "+ all groups": base + [c for cols in GROUPS.values() for c in cols]}
{k: len(v) for k, v in configs.items()}

{'baseline': 36,
 '+ alternation': 40,
 '+ in_session': 39,
 '+ relative': 40,
 '+ trend': 40,
 '+ hour_profile': 39,
 '+ all groups': 54}

## Corridas (2 modelos × 7 configuraciones × 5 semillas)

In [2]:
runs_csv = Path("data/reports/e8c_runs.csv")
if runs_csv.exists():  # reuse finished runs
    runs = pd.read_csv(runs_csv)
else:
    rows = []
    for target in TARGETS:
        for name, cols in configs.items():
            for seed in SEEDS:
                V.set_seed(seed)
                rows.append({"target": target, "config": name, "seed": seed,
                             "auc": V.auc(real, V.scores_for(real, mpud, cols, target), target)})
    runs = pd.DataFrame(rows)
    runs.to_csv(runs_csv, index=False)
mean = runs.pivot_table(index="config", columns="target", values="auc", aggfunc="mean")
spread = runs.groupby(["config", "target"])["auc"].agg(["min", "max"]).unstack()
gain = mean - mean.loc["baseline"]
pd.concat({"AUC (media)": mean, "ganancia": gain}, axis=1).loc[list(configs)].round(3)

AUC (media)        ganancia       
target                 y_f    y_s      y_f    y_s
config                                           
baseline             0.693  0.689    0.000  0.000
+ alternation        0.695  0.691    0.002  0.003
+ in_session         0.692  0.680   -0.000 -0.009
+ relative           0.701  0.684    0.009 -0.005
+ trend              0.697  0.692    0.005  0.003
+ hour_profile       0.702  0.691    0.009  0.002
+ all groups         0.702  0.690    0.009  0.001

In [3]:
spread.loc[list(configs)].round(3)

min           max       
target            y_f    y_s    y_f    y_s
config                                    
baseline        0.689  0.679  0.697  0.699
+ alternation   0.686  0.686  0.699  0.694
+ in_session    0.677  0.668  0.706  0.687
+ relative      0.692  0.669  0.725  0.694
+ trend         0.688  0.679  0.712  0.699
+ hour_profile  0.696  0.687  0.705  0.698
+ all groups    0.687  0.687  0.720  0.693

## Decisión por modelo

In [4]:
keep = {t: [g for g in GROUPS if gain.loc[f"+ {g}", t] >= 0.01] for t in TARGETS}
for t in TARGETS:
    print(f"{t}: keep {keep[t] or 'nothing'}; all groups {gain.loc['+ all groups', t]:+.3f}")

y_f: keep nothing; all groups +0.009
y_s: keep nothing; all groups +0.001


## Importancia (ganancia) con todos los grupos

Un modelo por patrón, preentrenado en MPUD y ajustado con todas las personas reales (3 de validación para la parada temprana). Solo descriptivo.

In [5]:
V.set_seed(0)
allc = configs["+ all groups"]
extra_cols = {c for cols in GROUPS.values() for c in cols}
imp = {}
for t in TARGETS:
    pre = T.pretrain_on_mpud(T.XGB, mpud.assign(y=mpud[t].astype(int)), allc)
    users = sorted(real.user_id.unique()); val = set(users[:3])
    r = real.assign(y=real[t].astype(int))
    tr, va = r[~r.user_id.isin(val)], r[r.user_id.isin(val)]
    model = T.XGB.finetune(pre, *T.xy(tr, allc), *T.xy(va, allc))
    g = pd.Series(model.get_score(importance_type="gain"))
    g.index = [allc[int(i[1:])] for i in g.index]
    imp[t] = (g / g.sum()).sort_values(ascending=False)
tables = []
for t in TARGETS:
    df = imp[t].head(15).rename_axis("feature").reset_index(name="share")
    df["new"] = df["feature"].isin(extra_cols)
    tables.append(df.round(3))
pd.concat(tables, axis=1, keys=TARGETS)

y_f                                y_s              
                  feature  share    new              feature  share    new
0              cur_logdur  0.132  False          cur_dur_pct  0.406   True
1            f60_minus_pf  0.131   True         s60_minus_ps  0.066   True
2             cur_dur_pct  0.066   True       cur_vs_typical  0.065   True
3                    n_15  0.060  False              prof_ps  0.036  False
4            logdur_sd_15  0.053  False             s60_rank  0.027  False
5                f60_rank  0.053  False         last_dur_log  0.022   True
6          cur_vs_typical  0.033   True          rate_s_hour  0.021   True
7         cur_survival_10  0.026   True      cur_survival_10  0.020   True
8                 few2_15  0.024  False         rmssd_gap_60  0.019   True
9          switch_rate_60  0.024   True       logdur_mean_15  0.018  False
10  prof_lognonuse_median  0.021  False    logdur_median_day  0.016  False
11                prof_pf  0.020  False        prof_freq_day  0.015  False
12            rate_f_hour  0.016   True  prof_use_day_logmin  0.014  False
13          prof_s_median  0.016  False             f60_rank  0.014  False
14     prof_logdur_median  0.014  False   prof_logdur_median  0.014  False

## Guardar

In [6]:
out = {"mean": mean.to_dict(), "gain": gain.to_dict(), "runs": runs.to_dict("records"), "keep": keep,
       "importance_top": {t: imp[t].head(20).to_dict() for t in TARGETS}}
Path("data/reports/e8c_feature_groups.json").write_text(json.dumps(out, indent=2, default=float))
print("-> data/reports/e8c_feature_groups.json")

-> data/reports/e8c_feature_groups.json
